## Assignment Topic: Twitter Entity Sentiment Analysis

### Source: Kaggle

### Objective: 
To perform sentiment analysis on text data using NLP techniques and machine learning models, and compare their performance.

In [1]:
# Importing the required Libraries
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Laharika\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Laharika\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### 1. Load Dataset into a DataFrame

In [2]:
# Load dataset
df = pd.read_csv("twitter_training.csv", header=None)

# Assign column names
df.columns = ["tweet_id", "entity", "sentiment", "tweet_content"]

# View first 5 rows
df.head()

,tweet_id,entity,sentiment,tweet_content
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


### 2. Data Understanding

In [3]:
# Shape of dataset
print("Shape:", df.shape)

Shape: (74682, 4)


In [4]:
# Check class distribution
print("\nSentiment Distribution:\n", df['sentiment'].value_counts())


Sentiment Distribution:
 sentiment
Negative      22542
Positive      20832
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [5]:
# Check missing values
print("\nMissing Values:\n", df.isnull().sum())


Missing Values:
 tweet_id           0
entity             0
sentiment          0
tweet_content    686
dtype: int64


In [6]:
#View Rows with Missing Values
df[df.isnull().any(axis=1)]

,tweet_id,entity,sentiment,tweet_content
61,2411,Borderlands,Neutral,NaN
553,2496,Borderlands,Neutral,NaN
589,2503,Borderlands,Neutral,NaN
745,2532,Borderlands,Positive,NaN
1105,2595,Borderlands,Positive,NaN
...,...,...,...,...
73972,9073,Nvidia,Positive,NaN
73973,9073,Nvidia,Positive,NaN
74421,9154,Nvidia,Positive,NaN
74422,9154,Nvidia,Positive,NaN


#### Observation: 
- Dataset contains rows with Irrelevant class as a sentiment label and could be removed in the data cleaning process.
- There are 686 rows with missing tweet content. Since text data is essential for sentiment analysis, these rows can be safely removed from the dataset.
- Tweets may contain noise which needs to be cleaned in the following steps.

### 3. Data Cleaning

In [7]:
# Drop rows with missing values(tweet_content)
df = df.dropna()

In [8]:
# Removing 'Irrelevant' class
df = df[df['sentiment'] != 'Irrelevant']

In [9]:
df['sentiment'].unique()

array(['Positive', 'Neutral', 'Negative'], dtype=object)

In [10]:
# Keep only required columns
df = df[['tweet_content', 'sentiment']]

In [11]:
# Reset index
df = df.reset_index(drop=True)

df.head()

,tweet_content,sentiment
0,im getting on borderlands and i will murder yo...,Positive
1,I am coming to the borders and I will kill you...,Positive
2,im getting on borderlands and i will kill you ...,Positive
3,im coming on borderlands and i will murder you...,Positive
4,im getting on borderlands 2 and i will murder ...,Positive


In [12]:
# Data shape after cleaning the dataset
df.shape

(61121, 2)

### 4. NLP Preprocessing

In [13]:
# Initialize tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [14]:
# Preprocessing function
def preprocess_text(text):
    
    text = text.lower()  # Lowercasing
    
    text = re.sub(r"http\S+", "", text)  # Remove URLs
    text = re.sub(r"@\w+", "", text)     # Remove mentions
    text = re.sub(r"#", "", text)     # Remove hashtag symbol from hashtags but keep the word
    text = re.sub(r"[^a-zA-Z]", " ", text)  # Remove special characters
    
    words = text.split()  # Tokenization
    
    words = [word for word in words if word not in stop_words]  # Remove stopwords
    words = [lemmatizer.lemmatize(word) for word in words]  # Lemmatization
    
    return " ".join(words)

In [15]:
#applying the pre-processing to tweets content
df['cleaned_text'] = df['tweet_content'].apply(preprocess_text)

df.head()

,tweet_content,sentiment,cleaned_text
0,im getting on borderlands and i will murder yo...,Positive,im getting borderland murder
1,I am coming to the borders and I will kill you...,Positive,coming border kill
2,im getting on borderlands and i will kill you ...,Positive,im getting borderland kill
3,im coming on borderlands and i will murder you...,Positive,im coming borderland murder
4,im getting on borderlands 2 and i will murder ...,Positive,im getting borderland murder


In [16]:
# Removing rows with no cleaned text
df = df[df['cleaned_text'].str.strip() != ""]

In [17]:
# final data shape after text-preprocessing
df.shape

(59653, 3)

### 5. Feature Engineering

In [18]:
# Define target
y = df['sentiment'] #carries the sentiment labels: Positive, Negative and Neutral

##### Implementing Vectorization Techniques

**1. Bag of Words (BoW)**

In [19]:
bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df['cleaned_text'])

**2. TF-IDF** 

In [20]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9
)

X_tfidf = tfidf.fit_transform(df['cleaned_text'])

### 6. Train–Validation–Test Split

In [21]:
# First split: Train (70%) and Temp (30%)
X_train_bow, X_temp_bow, y_train, y_temp = train_test_split(
    X_bow, y, test_size=0.3, random_state=42
)

X_train_tfidf, X_temp_tfidf, _, _ = train_test_split(
    X_tfidf, y, test_size=0.3, random_state=42
)

# Second split: Validation (15%) and Test (15%)
X_val_bow, X_test_bow, y_val, y_test = train_test_split(
    X_temp_bow, y_temp, test_size=0.5, random_state=42
)

X_val_tfidf, X_test_tfidf, _, _ = train_test_split(
    X_temp_tfidf, y_temp, test_size=0.5, random_state=42
)

In [22]:
# BoW Shapes
print("Bag of Words (BoW) Splits:")
print("X_train_bow shape:", X_train_bow.shape)
print("X_val_bow shape  :", X_val_bow.shape)
print("X_test_bow shape :", X_test_bow.shape)

# TF-IDF Shapes
print("\n TF-IDF Splits:")
print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_val_tfidf shape  :", X_val_tfidf.shape)
print("X_test_tfidf shape :", X_test_tfidf.shape)
print()

# target datashape
print("y_train shape:", y_train.shape)
print("y_val shape  :", y_val.shape)
print("y_test shape :", y_test.shape)

Bag of Words (BoW) Splits:
X_train_bow shape: (41757, 5000)
X_val_bow shape  : (8948, 5000)
X_test_bow shape : (8948, 5000)

 TF-IDF Splits:
X_train_tfidf shape: (41757, 10000)
X_val_tfidf shape  : (8948, 10000)
X_test_tfidf shape : (8948, 10000)

y_train shape: (41757,)
y_val shape  : (8948,)
y_test shape : (8948,)


**The dataset has been split into training, validation, and test sets to ensure proper model training, tuning, and unbiased evaluation.**

### 7. Model Training

#### 1. Logistic Regression

In [23]:
# BoW
lr_bow = LogisticRegression(max_iter=200)
lr_bow.fit(X_train_bow, y_train)
y_pred_lr_bow = lr_bow.predict(X_val_bow)

In [24]:
# TF-IDF
lr_tfidf = LogisticRegression(max_iter=200, C=2, solver='liblinear')
lr_tfidf.fit(X_train_tfidf, y_train)
y_pred_lr_tfidf = lr_tfidf.predict(X_val_tfidf)

C:\Users\Laharika\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


#### 2. Naive Bayes

In [25]:
# BoW
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
y_pred_nb_bow = nb_bow.predict(X_val_bow)

In [26]:
# TF-IDF
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
y_pred_nb_tfidf = nb_tfidf.predict(X_val_tfidf)

#### 3. Decision Tree

In [27]:
# BoW
dt_bow = DecisionTreeClassifier(max_depth=20)
dt_bow.fit(X_train_bow, y_train)
y_pred_dt_bow = dt_bow.predict(X_val_bow)

In [28]:
# TF-IDF
dt_tfidf = DecisionTreeClassifier(max_depth=20)
dt_tfidf.fit(X_train_tfidf, y_train)
y_pred_dt_tfidf = dt_tfidf.predict(X_val_tfidf)

### 8. Evaluation

In [29]:
## Evaluation function
def evaluate_model(y_true, y_pred, model_name):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    
    print(f"\n{model_name}")
    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    
    return accuracy, precision, recall, f1

In [30]:
results_list = []

# Logistic Regression
acc, prec, rec, f1 = evaluate_model(y_val, y_pred_lr_tfidf, "LR + TF-IDF")
results_list.append(["LR + TF-IDF", acc, prec, rec, f1])

acc, prec, rec, f1 = evaluate_model(y_val, y_pred_lr_bow, "LR + BoW")
results_list.append(["LR + BoW", acc, prec, rec, f1])


LR + TF-IDF
Accuracy : 0.8115
Precision: 0.8112
Recall   : 0.8115
F1 Score : 0.8108

LR + BoW
Accuracy : 0.7888
Precision: 0.7882
Recall   : 0.7888
F1 Score : 0.7882


In [31]:
# Naive Bayes
acc, prec, rec, f1 = evaluate_model(y_val, y_pred_nb_bow, "NB + BoW")
results_list.append(["NB + BoW", acc, prec, rec, f1])

acc, prec, rec, f1 = evaluate_model(y_val, y_pred_nb_tfidf, "NB + TF-IDF")
results_list.append(["NB + TF-IDF", acc, prec, rec, f1])


NB + BoW
Accuracy : 0.7223
Precision: 0.7217
Recall   : 0.7223
F1 Score : 0.7199

NB + TF-IDF
Accuracy : 0.7588
Precision: 0.7611
Recall   : 0.7588
F1 Score : 0.7557


In [32]:
# Decision Tree
acc, prec, rec, f1 = evaluate_model(y_val, y_pred_dt_tfidf, "DT + TF-IDF")
results_list.append(["DT + TF-IDF", acc, prec, rec, f1])

acc, prec, rec, f1 = evaluate_model(y_val, y_pred_dt_bow, "DT + BoW")
results_list.append(["DT + BoW", acc, prec, rec, f1])


DT + TF-IDF
Accuracy : 0.8352
Precision: 0.8352
Recall   : 0.8352
F1 Score : 0.8351

DT + BoW
Accuracy : 0.8555
Precision: 0.8556
Recall   : 0.8555
F1 Score : 0.8555


### 9. Comparison Table

In [33]:
results_df = pd.DataFrame(results_list, 
                          columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score"])

# Sort by best model
results_df = results_df.sort_values(by="F1 Score", ascending=False)

results_df

,Model,Accuracy,Precision,Recall,F1 Score
5,DT + BoW,0.855498,0.855572,0.855498,0.855529
4,DT + TF-IDF,0.835159,0.835225,0.835159,0.835145
0,LR + TF-IDF,0.811466,0.811229,0.811466,0.810790
1,LR + BoW,0.788780,0.788191,0.788780,0.788209
3,NB + TF-IDF,0.758829,0.761114,0.758829,0.755698
2,NB + BoW,0.722284,0.721725,0.722284,0.719917


### Best Performing Model and Insights

In [35]:
best_model_name = results_df.iloc[0]["Model"]
print("Best Model:", best_model_name)

Best Model: DT + BoW


- Decision Tree with Bag of Words achieved the highest performance due to its ability to capture non-linear relationships and memorize frequent patterns in the dataset.

- Logistic Regression with TF-IDF did not outperform Decision Tree because TF-IDF reduces the importance of frequently occurring words, which in this dataset carry strong sentiment signals.

- Additionally, Logistic Regression assumes linear separability, which may not be sufficient for capturing complex patterns in short and repetitive tweet data.

- While Decision Tree achieved the highest score, Logistic Regression with TF-IDF is preferred for real-world applications due to better generalization and stability.